# Diffusion Models- How AI Creates Images from Noise

**Module 6 · Lesson 6.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Generation is denoising - the big idea
- The forward process and the noise schedule
- Train the denoiser - predict the noise
- Sampling - reverse the process, step by step
- Latent diffusion and text conditioning - how Stable Diffusion works
- The 2026 landscape and practice

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install torch -q

# Reproducibility - the lesson uses seeded random values throughout

## The whole game: turn noise into an image by removing it, a slice at a time

> 🎨 **Analogy**
>
> **Picture a sculptor with a block of marble.** The figure is "already inside" - the sculptor just removes what does not belong, chip by chip, until it emerges. A diffusion model generates the same way: it starts from a block of pure noise and removes a predicted slice of it at every step, and a coherent image appears from the static.
> That is the root idea of every model in this lesson: **we never teach a network to draw - we teach it to undo noise**. Train it to look at a noisy image and say "this much of what you see is noise", subtract that, and repeat. Do it enough times, starting from nothing but static, and you have generated an image.
> **Where the analogy breaks down:** a sculptor has the finished figure in mind from the first chip. The model has no global plan - at each step it only predicts "what noise to remove next" from the current noisy image and your text prompt. That blindness is exactly why guidance and many small steps matter, as you will see in Steps 4 and 5.

We build this bottom-up in PyTorch so nothing is a black box, then connect it to the production stack - latent diffusion, the `diffusers` library, and the 2026 frontier of Diffusion Transformers and flow matching. The image primitives here are the foundation the rest of Module 6 stands on: the Stable Diffusion ecosystem in Lesson 6.2, controlled generation in Lesson 6.3, and the CLIP text-image space in Lesson 6.4.

- **Explain** the forward (noising) and reverse (denoising) processes, and why image generation is framed as iterative denoising rather than drawing.

- **Implement** the core of DDPM - add noise on a schedule, train a network to predict that noise with an MSE loss, and sample by reversing the process.

- **Describe** latent diffusion and text conditioning - a VAE latent space, a text encoder, and cross-attention - to explain how Stable Diffusion actually works.

- **Compare** the 2026 landscape - U-Net versus Diffusion Transformer, DDPM versus flow matching, and few-step distillation - and choose a model and sampler for a task.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier modules. Each one is load-bearing for today.

## The setup: a tiny image and a noise mixer we reuse all lesson

Everything runs in plain **PyTorch**. We use a tiny 64x64 grayscale image and one helper, `add_noise`, that mixes an image with Gaussian noise by a ratio. That single mix is the whole forward process.

**📝 `setup.py`** - *PyTorch*

In [ ]:
import torch

def make_sample_image(size: int = 64) -> torch.Tensor:
    """A toy 'image': a white square on black. A real one would load from disk."""
    img = torch.zeros(1, size, size)   # 1 channel, 64x64, all black
    img[0, 16:48, 16:48] = 1.0      # a white square in the middle
    return img

def add_noise(image: torch.Tensor, alpha_bar: torch.Tensor):
    """Mix image and noise. alpha_bar in [0,1] is how much IMAGE survives."""
    noise = torch.randn_like(image)                  # Gaussian noise, same shape
    noisy = torch.sqrt(alpha_bar) * image + torch.sqrt(1 - alpha_bar) * noise
    return noisy, noise                              # return the noise too - it is the 'answer'

image = make_sample_image()
noisy, noise = add_noise(image, torch.tensor(0.5))
print(image.shape, noisy.shape)
# Output: torch.Size([1, 64, 64]) torch.Size([1, 64, 64])
print(f"alpha_bar=0.5 -> half image, half noise")
# Output: alpha_bar=0.5 -> half image, half noise

- `make_sample_image` - a stand-in for a real photo so the math stays visible. Swap in MNIST or your own images and nothing else changes.

- `add_noise` is the entire forward process: `sqrt(alpha_bar)` of the image plus `sqrt(1 - alpha_bar)` of fresh Gaussian noise. The coefficients are square roots because their *squares* are variances: `alpha_bar + (1 - alpha_bar) = 1`, so a unit-variance image plus unit-variance noise stays variance-1 at every `t` (nothing blows up or fades). So when we say "% noise" we mean the variance share `1 - alpha_bar`; the amplitude actually mixed in is its square root.

- It returns **the noise as well as the noisy image**. That returned noise is the label we will train the network to predict in Step 3 - keep an eye on it.

- `alpha_bar` (written `ᾱ`) is the one knob: 1.0 means a clean image, 0.0 means pure noise. Step 2 turns it into a schedule over many timesteps.

---

## 🎯 Concept 1: Generation is denoising - the big idea

### Generation is denoising - the big idea

Two processes: one destroys an image with noise, the other rebuilds it. Generation is the second.

**A photo developing in a tray.** It starts as a grey blank and an image swims into focus over a minute. Run that film backwards and a clear photo dissolves into grey. Diffusion has both directions: forward dissolves an image into static; reverse develops static into an image.

The gain: forward = destroy, reverse = create, and they are the same path run opposite ways. No drawing anywhere - just adding or removing noise.

**📝 `forward_process.py`** - *PyTorch*

In [ ]:
# THE FORWARD PROCESS: walk an image from clean to pure noise.
# x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise

alpha_bars = [0.99, 0.80, 0.50, 0.20, 0.05, 0.01]
for ab in alpha_bars:
    noisy, _ = add_noise(image, torch.tensor(ab))
    pct = round(100 * (1 - ab))
    print(f"alpha_bar={ab:.2f}  ->  ~{pct:>2d}% noise")

# Output: alpha_bar=0.99  ->  ~ 1% noise   (square clearly visible)
# Output: alpha_bar=0.80  ->  ~20% noise   (square slightly grainy)
# Output: alpha_bar=0.50  ->  ~50% noise   (square fading)
# Output: alpha_bar=0.20  ->  ~80% noise   (barely there)
# Output: alpha_bar=0.05  ->  ~95% noise   (nearly gone)
# Output: alpha_bar=0.01  ->  ~99% noise   (pure static)

- As `alpha_bar` drops from 0.99 to 0.01, the same image goes from crisp to pure static - the forward process, in six snapshots.

- There is **no neural network and no learning here** - it is fixed arithmetic. That matters: the "hard" part is only ever the reverse direction.

- The end state at `alpha_bar=0.01` is essentially indistinguishable from `torch.randn`. That is precisely where generation will *start* in Step 4.

- Run the interactive below to feel both directions before we write any training code.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  Diffusion has two processes. The **forward** process is fixed arithmetic that destroys an image by mixing in more and more Gaussian noise. The **reverse** process - the only learned part - removes a little predicted noise at a time. Generation runs the reverse process starting from pure static, so an image appears "from nothing". Everything else in this lesson is detail on those two ideas.

---

## 🎯 Concept 2: The forward process and the noise schedule

### The forward process and the noise schedule

How much noise to add at each step - and why the choice of schedule changes image quality.

**Dimming a light over a minute.** You could yank it from full to off in the first two seconds (everyone sits in the dark for 58 seconds) or ease it down smoothly. The schedule is the dimmer curve - it sets how fast "image" fades to "dark" across the timesteps.

The gain: the schedule is a curve you choose, not a constant. A gentler curve keeps useful signal alive longer.

- **β~t~** (beta) - the small amount of noise added at step *t*. The schedule is really a choice of these betas.

- **α~t~ = 1 - β~t~** - the fraction of the signal that survives a single step.

- **ᾱ~t~** (alpha-bar) - the product of all the per-step survivals, so it is how much of the original image is left after *t* steps. This is the one number `add_noise` needs.

**📝 `noise_schedule.py`** - *PyTorch*

In [ ]:
def linear_schedule(T=1000, beta_start=1e-4, beta_end=0.02):
    """The original DDPM schedule: betas ramp up linearly."""
    return torch.linspace(beta_start, beta_end, T)

def cosine_schedule(T=1000):
    """Improved DDPM: signal fades on a gentler cosine curve."""
    steps = torch.arange(T + 1) / T
    abar = torch.cos((steps + 0.008) / 1.008 * torch.pi / 2) ** 2
    abar = abar / abar[0]
    betas = 1 - (abar[1:] / abar[:-1])
    return betas.clamp(max=0.999)

def alpha_bars_from(betas):
    return torch.cumprod(1.0 - betas, dim=0)   # cumulative product

T = 1000
lin = alpha_bars_from(linear_schedule(T))
cos = alpha_bars_from(cosine_schedule(T))
print(f"step  50: linear abar={lin[50]:.3f}  cosine abar={cos[50]:.3f}")
print(f"step 500: linear abar={lin[500]:.3f}  cosine abar={cos[500]:.3f}")
# Output: step  50: linear abar=0.970  cosine abar=0.992
# Output: step 500: linear abar=0.078  cosine abar=0.492

- `linear_schedule` ramps the per-step noise from tiny to small linearly - the original DDPM choice. `cosine_schedule` follows a cosine curve so signal is preserved longer through the middle timesteps.

- `alpha_bars_from` turns per-step betas into the cumulative `alpha_bar` with one `torch.cumprod` - that is the survival product from the formula above.

- Look at **step 500**: linear has fallen to a thin sliver (`abar=0.078`, under a tenth of the signal), while cosine still keeps roughly half (`abar=0.492`). Linear crashes toward pure noise far earlier - its later timesteps carry little signal to learn from.

- That gap is the whole argument for cosine: more timesteps carry useful signal, so the denoiser learns from more of them.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  The noise schedule decides how fast signal dies across the `T` timesteps. The original **linear** schedule destroys the image too quickly in the middle; the **cosine** schedule fades it more evenly, so more timesteps carry learnable signal. The key quantity is `alpha_bar` at each step - how much original image survives - and getting its curve right measurably improves sample quality.

---

## 🎯 Concept 3: Train the denoiser - predict the noise

### Train the denoiser - predict the noise

One network, one trick: it predicts the noise that was added, not the clean image. The loss is plain MSE.

**An audio engineer cleaning a recording.** They do not try to re-record the clean voice - they listen for the *hiss* and subtract it. Identifying "what is the noise here" is a far easier job than reconstructing the original from scratch.

The gain: the network predicts noise, and we subtract it. Easier target, simpler loss, more stable training.

**📝 `denoiser_and_training.py`** - *PyTorch*

In [ ]:
import torch.nn as nn

class SimpleDenoiser(nn.Module):
    """Predicts the noise in a noisy image. A real model uses a U-Net or a DiT;
    this CNN captures the idea in a few layers."""
    def __init__(self, ch=1, hidden=64, T=1000):
        super().__init__()
        self.time_embed = nn.Embedding(T, hidden)        # timestep -> vector (lookup table)
        # two NAMED blocks so we can inject the timestep cleanly between them
        self.in_conv = nn.Sequential(nn.Conv2d(ch, hidden, 3, padding=1), nn.ReLU())
        self.out_net = nn.Sequential(
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, ch, 3, padding=1),         # out: predicted noise
        )

    def forward(self, x, t):
        emb = self.time_embed(t).unsqueeze(-1).unsqueeze(-1)  # (B,hidden,1,1)
        h = self.in_conv(x) + emb        # add the timestep vector to every pixel (broadcast)
        return self.out_net(h)           # run the rest of the network

model = SimpleDenoiser(T=T)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
abar = alpha_bars_from(linear_schedule(T))

for step in range(300):
    x0 = torch.stack([make_sample_image() for _ in range(16)])   # (16,1,64,64)
    t = torch.randint(0, T, (16,))                            # a random timestep per image
    ab = abar[t].view(-1, 1, 1, 1)
    noise = torch.randn_like(x0)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise   # forward process

    pred = model(x_t, t)                                   # predict the noise
    loss = nn.functional.mse_loss(pred, noise)             # compare to the REAL noise
    opt.zero_grad(); loss.backward(); opt.step()
    if (step + 1) % 100 == 0:
        print(f"step {step+1:3d}  loss {loss.item():.4f}")

# Output: step 100  loss 0.4117
# Output: step 200  loss 0.2098
# Output: step 300  loss 0.1413

- **Timestep embedding** (`time_embed`) tells the network *how noisy* this image is, so it can scale its prediction - low `t` means predict little noise, high `t` means a lot. Here `nn.Embedding` is a learned lookup table: one trainable vector per timestep `0..T-1` (the same idea as a text embedding from Lesson 4.1, but keyed by the integer step). `forward` adds that vector to the first conv's output, broadcasting it across every pixel, so the whole image "knows" its noise level.

- Each step: take clean images, pick a random timestep per image, run the **forward process** to make `x_t`, and keep the exact `noise` we added.

- The network predicts the noise; `mse_loss(pred, noise)` compares it to the real noise. **That MSE is the whole objective** - no adversary, no special loss.

- `opt.zero_grad(); loss.backward(); opt.step()` is the standard PyTorch update from Module 4: clear last step's gradients, compute new ones from the loss, then nudge the weights downhill. That trio is what actually makes the loss fall.

- The falling loss means the network is getting better at spotting noise at every level. That is all "training a diffusion model" is.

#### 💡 What just happened

⚡ What just happened?
  We trained the only learned component: a network that takes a noisy image plus its timestep and predicts the noise. The loop is take a clean image, add known noise, ask the network what noise it sees, and minimize the MSE against the truth. The timestep input lets one network handle every noise level. **That is the complete DDPM training algorithm** - everything bigger (U-Net, then DiT) is the same loop with a heavier network.

---

## 🎯 Concept 4: Sampling - reverse the process, step by step

### Sampling - reverse the process, step by step

Start from noise, subtract predicted noise repeatedly, and trade steps for speed with DDIM and guidance.

**Cleaning a foggy windscreen with one wipe versus many.** One big swipe smears it; a dozen light passes leave it clear. Each pass works on what the last one left, so the result keeps improving.

The gain: sampling is a loop of small denoising passes. More passes, cleaner image - up to a point.

**📝 `sampling.py`** - *PyTorch*

In [ ]:
@torch.no_grad()
def sample(model, T=1000, size=64):
    """Generate one image by denoising pure noise from t=T-1 down to t=0."""
    betas = linear_schedule(T)
    alphas = 1.0 - betas
    abar = torch.cumprod(alphas, dim=0)

    x = torch.randn(1, 1, size, size)             # START: pure noise
    for t in reversed(range(T)):
        pred_noise = model(x, torch.tensor([t]))    # what noise is in x?
        a, ab, b = alphas[t], abar[t], betas[t]
        z = torch.randn_like(x) if t > 0 else 0  # no fresh noise on last step
        # one DDPM reverse step: remove a slice of predicted noise
        x = (1 / torch.sqrt(a)) * (x - (b / torch.sqrt(1 - ab)) * pred_noise) \
            + torch.sqrt(b) * z
    return x.clamp(0, 1)

img = sample(model, T=T)
print(img.shape, f"range [{img.min():.2f}, {img.max():.2f}]")
# Output: torch.Size([1, 1, 64, 64]) range [0.00, 1.00]
# A square emerges from noise - we trained on squares, so it generates squares.

- **Start from `torch.randn`** - the same pure static the forward process ended on. The model never sees a real image to begin from.

- Each iteration predicts the noise, then the DDPM update **removes only a slice** (scaled by `beta`) and adds a touch of fresh noise `z` to keep the process stochastic - except the final step, which is clean.

- Looping `reversed(range(T))` walks from `t=T-1` (noisy) down to `t=0` (clean). This is hundreds to a thousand network calls - the latency Step 1 warned about.

- **DDIM** rewrites this update as a deterministic step you can take in larger jumps, reaching similar quality in ~20-50 steps instead of ~1000. The animation below makes that trade visible.

> 📦 **Info**
>
> 🎯 Classifier-free guidance (CFG) - making the prompt actually steer
> Conditioning aside (Step 5), a plain text-conditioned model barely follows the prompt. **CFG** runs the denoiser twice per step - once with the prompt, once without - and amplifies the difference:
> noise = noise~uncond~ + scale × (noise~cond~ - noise~uncond~)noise~cond~ = the with-prompt prediction, noise~uncond~ = the without-prompt one; scale ~7-8 is a common default
> Read it straight off the formula: scale **0** gives the unconditional prediction (ignores the prompt); scale **1** gives the plain conditional (follows the prompt, un-amplified); a moderate scale (~7-8) amplifies prompt adherence; a very high scale over-cooks the image. It is the single most important sampling knob after step count. (The toy `sample()` above has no prompt input, so it cannot do this - CFG arrives once text conditioning does, in Step 5.)
> **In plain words:** CFG is a "listen to my words harder" dial. The model imagines the image twice - once paying attention to your prompt, once ignoring it - and then exaggerates the difference. Turn it up and the picture follows your prompt more; turn it too far and it looks burnt.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?
  Sampling starts from pure noise and applies the trained denoiser in a loop, removing a slice of predicted noise each step until an image emerges. **DDPM** uses many steps; **DDIM** and solvers like DPM-Solver reach similar quality in far fewer; **distillation** pushes it to a handful. **Classifier-free guidance** is what makes the text prompt steer the result strongly. Step count and guidance scale are the two dials you actually turn in production.

---

## 🎯 Concept 5: Latent diffusion and text conditioning - how Stable Diffusion works

### Latent diffusion and text conditioning - how Stable Diffusion works

Run diffusion in a compressed latent space, and let a text encoder steer it through cross-attention.

**Editing the thumbnail, not the billboard.** Reworking a giant billboard pixel by pixel is slow; sketch on a small thumbnail, then blow it up. Latent diffusion denoises in "thumbnail" space and the VAE blows the result back up to full resolution.

The gain: diffuse in a compressed latent, decode once at the end. Same algorithm, a fraction of the pixels.

**📝 `stable_diffusion.py`** - *diffusers*

<!-- Illustrative: requires the diffusers library, a gated Stable Diffusion 3.5 checkpoint, and a CUDA GPU. Shown for reference; not run here. -->

```python
# pip install diffusers transformers accelerate torch
import torch
from diffusers import StableDiffusion3Pipeline

# SD 3.5: a Diffusion Transformer trained with rectified flow (Step 6).
# The diffusers API is the same shape for SDXL, SD3.5, and Flux.
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-large",
    torch_dtype=torch.bfloat16,
).to("cuda")

image = pipe(
    prompt="a red origami crane on a wooden desk, soft window light, photographic",
    negative_prompt="blurry, distorted, extra folds",
    num_inference_steps=28,     # the speed/quality dial from Step 4
    guidance_scale=4.5,         # CFG; SD3.5 likes a lower scale than SD1.5
    height=1024, width=1024,
    generator=torch.Generator("cuda").manual_seed(0),
).images[0]
image.save("crane.png")
print(image.size)
# Output: (1024, 1024)
```


- Under `pipe(...)` sits everything you built: a noise schedule, a denoiser, and the sampling loop from Step 4 - plus a **VAE** that keeps diffusion in latent space and a **text encoder** that turns your prompt into embeddings.

- **Text conditioning** works through *cross-attention*: the prompt embeddings (the same kind of embedding from Lesson 4.1) are injected into the denoiser at every layer, so the predicted noise depends on your words. CLIP and that embedding space get a full treatment in Lesson 6.4.

- `num_inference_steps` and `guidance_scale` are the exact Step-4 dials - the library just exposes them. Note SD3.5 prefers a lower guidance scale than older SD1.5 did.

- `negative_prompt` lists what to steer *away* from - it fills CFG's unconditional branch (Step 4), so generation is pushed away from "blurry, distorted, extra folds" rather than those words being hard-blocked.

- Swap `StableDiffusion3Pipeline` for `FluxPipeline` and a Flux checkpoint and almost nothing else changes - the scheduler is decoupled from the denoiser. On the DDPM-trained line (SD1.5/SDXL) one denoiser runs with DDPM, DDIM, or DPM-Solver; flow-matching models like SD3.5 and Flux use flow-matching (ODE) samplers instead.

#### 💡 What just happened

⚡ What just happened?
  Stable Diffusion is the toy DDPM you built plus two production pieces. **Latent diffusion** runs the whole process in a compressed VAE latent - a fraction of the pixels - which is what makes high-resolution generation affordable. **Text conditioning** feeds prompt embeddings into the denoiser through cross-attention, so the noise prediction follows your words. The `diffusers` library wraps all of it behind one call, with the same step-count and guidance dials you already understand.

---

## 🎯 Concept 6: The 2026 landscape and practice

### The 2026 landscape and practice

U-Net to Diffusion Transformer, DDPM to flow matching, many steps to a few - plus how to evaluate and ship responsibly.

**Same destination, straighter road, bigger engine.** Flow matching is a straighter route from noise to image (fewer turns, fewer steps); the Diffusion Transformer is a bigger engine that scales with data. You are still driving from "noise" to "image" - the route and the engine got better.

The gain: the mental model carries over; only the backbone and the path changed. Denoising in, image out - still true in 2026.

| Model | Backbone | Training path | What it changed |
|---|---|---|---|
| SD 1.5 / SDXL | U-Net + VAE | DDPM | First widely-open latent diffusion; SDXL pushed to 1024px. |
| SD 3.5 | DiT (MMDiT) | Rectified flow | Transformer backbone + flow matching (MMDiT = a multimodal DiT variant); strong open-weight line. |
| Flux (Flux.1 / Flux 2) | DiT | Flow matching | ~12B-parameter open-weight leader; best text rendering. |
| Distilled (LCM, SD3.5 Large Turbo) | DiT / U-Net | Consistency / distillation | 1-4 step generation, near real-time (LCM = Latent Consistency Model). |

> 📦 **Info**
>
> 🌱 Three shifts that date the "U-Net DDPM" picture
> - **Diffusion Transformer (DiT):** replace the U-Net with transformer blocks over image patches. Scales better with data and compute; the backbone of SD3.5, Flux, and video models. (The transformer is the same family from Module 4.)
> - **Flow matching / rectified flow:** instead of the DDPM stochastic process, learn a near-straight path from noise to data. Faster convergence, fewer sampling steps. SD3 and Flux train this way.
> - **Consistency / distillation:** distil the many-step sampler into 1-4 steps for near-real-time generation, shifting the latency-quality frontier.

- **FID** (Frechet Inception Distance) is the standard quality metric - lower is better - comparing the distribution of generated images to real ones. It captures quality and diversity but not prompt adherence, so it is paired with a text-image alignment score (CLIP-based; Lesson 6.4).

- **Provenance:** generated images should carry content-credential metadata or an invisible watermark (the C2PA standard, and model-specific schemes) so downstream systems can tell AI-generated media apart - increasingly a legal as well as ethical requirement.

- **Safety:** production pipelines keep a safety checker on outputs; the deepfake and consent risks of photoreal generation are real, and we treat responsible generation in depth in Module 15.

- **Choosing:** need speed and interactivity -> a distilled few-step model; need maximum fidelity and text rendering -> a full Flux / SD3.5 run at ~25-30 steps with a tuned guidance scale.

#### 💡 What just happened

⚡ What just happened?
  The intuition you built is timeless; the frontier moved on two axes. The denoiser backbone went from **U-Net to Diffusion Transformer**, and the training path went from **DDPM to flow matching / rectified flow** - together giving Flux and SD3.5 their jump in prompt following and text rendering. **Distillation** brought few-step, near-real-time generation. You evaluate with FID plus a text-alignment score, and ship with provenance and safety in place.

## Synthesis: from a toy DDPM to a real text-to-image call

### The complete picture, in one breath

A diffusion model **generates by denoising**. The **forward process** destroys an image by adding Gaussian noise on a **schedule** (cosine keeps signal alive longer than linear). A single network is trained to **predict that noise** with a plain MSE loss - the one trick that makes it stable. **Sampling** reverses the process from pure static, a slice at a time, with **step count** and **guidance scale** as the dials. **Latent diffusion** runs all of that in a compressed VAE space and steers it with **text embeddings through cross-attention** - that is Stable Diffusion. In 2026 the backbone is a **Diffusion Transformer**, the path is **flow matching**, and **distillation** makes it fast - but the idea you coded is unchanged.

> ℹ️ **Info**
>
> Where this goes next in Module 6
> - **Lesson 6.2 - the Stable Diffusion ecosystem:** checkpoints, community models, and fine-tuning images with LoRA and DreamBooth.
> - **Lesson 6.3 - controlled generation:** ControlNet, IP-Adapter, inpainting and img2img - steering composition, not just the prompt.
> - **Lesson 6.4 - ViT and CLIP:** the text-image embedding space that the text encoder here relies on.
> - **Lesson 6.6 - video and 3D:** the same denoising idea extended across time and space.

- Ho et al., *Denoising Diffusion Probabilistic Models* (DDPM, 2020) - [arxiv.org/abs/2006.11239](https://arxiv.org/abs/2006.11239)

- Song et al., *Denoising Diffusion Implicit Models* (DDIM, 2020) - [arxiv.org/abs/2010.02502](https://arxiv.org/abs/2010.02502)

- Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models* (2022) - [arxiv.org/abs/2112.10752](https://arxiv.org/abs/2112.10752)

- Ho & Salimans, *Classifier-Free Diffusion Guidance* (2022) - [arxiv.org/abs/2207.12598](https://arxiv.org/abs/2207.12598)

- Peebles & Xie, *Scalable Diffusion Models with Transformers* (DiT, 2023) - [arxiv.org/abs/2212.09748](https://arxiv.org/abs/2212.09748)

- Esser et al., *Scaling Rectified Flow Transformers for High-Resolution Image Synthesis* (SD3, 2024) - [arxiv.org/abs/2403.03206](https://arxiv.org/abs/2403.03206)

- Hugging Face *diffusers* library and schedulers guide - [huggingface.co/docs/diffusers](https://huggingface.co/docs/diffusers)

## Recap

> ✅ **Info**
>
> What we built
> - **Generation is denoising** - a forward process destroys an image with noise; the learned reverse process rebuilds it.
> - **Noise schedule** - `alpha_bar` over T timesteps; cosine preserves signal longer than linear.
> - **Train the denoiser** - predict the added noise with MSE; the timestep input handles every noise level.
> - **Sampling** - reverse from pure noise; DDIM and solvers cut the step count; CFG makes the prompt steer.
> - **Latent diffusion + text conditioning** - a VAE latent and cross-attention text encoder; that is Stable Diffusion via `diffusers`.
> - **2026 landscape** - DiT backbone, flow matching, distillation, FID + provenance + safety.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Diffusion Models- How AI Creates Images from Noise**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-6_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-6.1.html` - regenerate this notebook whenever the source HTML is updated.*
